# Fine-tune Qwen with LoRA
Does a quick test by default. For the real run put `TEST_RUN=0` in front of the command (README step 5).

In [1]:
import os
import time
import torch
import datasets
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments, TrainerCallback
from transformers.trainer_utils import get_last_checkpoint
from peft import LoraConfig, get_peft_model, PeftModel

MODEL_DIR = "../model/Qwen3-0.6B-Base"
DATA = "../data/train_test/"
OUT = "../model/"

TEST_RUN = os.environ.get("TEST_RUN", "1") != "0"
BATCH = int(os.environ.get("BATCH", "1"))  # 1 on the laptop (6GB), 4 on a 3090
STEP = 8 if TEST_RUN else 32  # pieces per training step
SAVE_EVERY = 5 if TEST_RUN else 250  # 250 steps is ~50 min on the laptop
LR = 2e-4

RUN_DIR = OUT + ("lora_test_runs/" if TEST_RUN else "lora_runs/")  # checkpoints
FINAL = OUT + ("lora_test" if TEST_RUN else "lora")
LOG = OUT + ("train_log_test.txt" if TEST_RUN else "train_log.txt")

W1118 13:25:31.451000 25456 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [2]:
# the model takes ~1.3 GB and each piece in a batch can need up to ~3 GB
free = torch.cuda.mem_get_info()[0] / 1e9
need = 1.5 + 3 * BATCH
print("test run" if TEST_RUN else "real run", f"| batch {BATCH} | {free:.1f} GB free, need ~{need:.0f} GB")
assert free > need, f"need ~{need:.0f} GB free GPU memory, use the other GPU or a smaller BATCH"
assert STEP % BATCH == 0, f"BATCH has to divide {STEP}"

def log(msg):
    print(msg)
    with open(LOG, "a") as f:
        f.write(time.strftime("%H:%M:%S") + " " + msg + "\n")

test run | batch 1 | 5.4 GB free, need ~4 GB


## Load the data

In [3]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)

cols = ['input_ids', 'attention_mask']
train = datasets.Dataset.from_parquet(DATA + 'train.parquet').select_columns(cols)
val = datasets.Dataset.from_parquet(DATA + 'val.parquet').select_columns(cols)

# random val pieces for checking, the first rows of the file are all pdp
val_sample = val.shuffle(seed=42).select(range(100 if TEST_RUN else 300))
print(len(train), "train pieces,", len(val_sample), "val pieces for checking")

89344 train pieces, 100 val pieces for checking


## Batches
Qwen uses `<|endoftext|>` for padding and for the end of a poem. The normal collator would hide both from the
loss and the model would never learn to end a poem, so the labels come from the attention mask instead.

In [4]:
def collate(batch):
    longest = max(len(x['input_ids']) for x in batch)
    ids, mask, labels = [], [], []
    for x in batch:
        pad = longest - len(x['input_ids'])
        ids.append(x['input_ids'] + [tokenizer.pad_token_id] * pad)
        mask.append([1] * len(x['input_ids']) + [0] * pad)
        labels.append(x['input_ids'] + [-100] * pad)
    return {'input_ids': torch.tensor(ids), 'attention_mask': torch.tensor(mask), 'labels': torch.tensor(labels)}

# quick check: real tokens are in the labels, padding isn't
b = collate([train[0], train[1], train[2]])
real = b['attention_mask'] == 1
print("ok" if (b['labels'][real] == b['input_ids'][real]).all() and (b['labels'][~real] == -100).all() else "WRONG")

ok


## Model

In [5]:
model = AutoModelForCausalLM.from_pretrained(MODEL_DIR, dtype=torch.bfloat16).to('cuda')
model.config.use_cache = False

lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type='CAUSAL_LM',
                  target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'])
model = get_peft_model(model, lora)
model.print_trainable_parameters()

Loading weights: 100%|██████████| 310/310 [00:00<00:00, 380.69it/s]


trainable params: 10,092,544 || all params: 606,142,464 || trainable%: 1.6650


## Header check
Loss on val poems with the real `Author:` name vs `Author: Unknown` (only the poem counts, not the header).
If the model is learning the poets, the gap goes up.

In [6]:
val_text = pd.read_parquet(DATA + 'val.parquet', columns=['text', 'header'])
named = val_text[val_text['header'] != 'Unknown'].sample(50 if TEST_RUN else 200, random_state=42)

def poem_loss(model, name, poem):
    head = tokenizer(f"Author: {name}\n\n")['input_ids']
    ids = tokenizer(f"Author: {name}\n\n" + poem)['input_ids']
    if ids[:len(head)] != head:
        return None
    x = torch.tensor([ids]).to('cuda')
    labels = x.clone()
    labels[0, :len(head)] = -100  # don't count the header
    with torch.no_grad(), torch.autocast('cuda', dtype=torch.bfloat16):
        return model(input_ids=x, labels=labels).loss.item()

def header_check(model):
    model.eval()
    real, unknown = [], []
    for text, name in zip(named['text'], named['header']):
        poem = text.split('\n\n', 1)[1]
        r, u = poem_loss(model, name, poem), poem_loss(model, 'Unknown', poem)
        if r is not None and u is not None:
            real.append(r)
            unknown.append(u)
    model.train()
    r, u = sum(real) / len(real), sum(unknown) / len(unknown)
    return f"real name {r:.3f}, Unknown {u:.3f}, gap {u - r:+.3f}"

log("header check before training: " + header_check(model))

header check before training: real name 4.198, Unknown 4.225, gap +0.026


## Train
Checkpoint every `SAVE_EVERY` steps (keeps the last 3) with the val loss and the header check, all in the log file.
If there's a checkpoint already it carries on from there.

In [7]:
class Progress(TrainerCallback):
    times = []
    def on_step_end(self, args, state, control, **kwargs):
        self.times.append(time.time())
    def on_log(self, args, state, control, logs=None, **kwargs):
        log(f"step {state.global_step}/{state.max_steps} {logs}")
    def on_save(self, args, state, control, model=None, **kwargs):
        log(f"checkpoint {state.global_step}: " + header_check(model))

progress = Progress()

def make_trainer(max_steps):
    args = TrainingArguments(
        output_dir=RUN_DIR,
        per_device_train_batch_size=BATCH,
        per_device_eval_batch_size=BATCH,
        gradient_accumulation_steps=STEP // BATCH,
        learning_rate=LR,
        lr_scheduler_type='cosine',
        warmup_steps=2 if TEST_RUN else 100,
        num_train_epochs=1,
        max_steps=max_steps,  # -1 = the whole pass
        bf16=True,
        logging_steps=2 if TEST_RUN else 10,
        eval_strategy='steps',
        eval_steps=SAVE_EVERY,
        save_strategy='steps',
        save_steps=SAVE_EVERY,
        save_total_limit=3,
        report_to='none',
    )
    return Trainer(model=model, args=args, train_dataset=train, eval_dataset=val_sample,
                   data_collator=collate, callbacks=[progress])

trainer = make_trainer(10 if TEST_RUN else -1)
last = get_last_checkpoint(RUN_DIR) if os.path.isdir(RUN_DIR) else None
if last:
    log("carrying on from " + last)
else:
    log(f"starting, val loss {trainer.evaluate()['eval_loss']:.3f}")
trainer.train(resume_from_checkpoint=last)

step 0/0 {'eval_loss': 4.273749828338623, 'eval_model_preparation_time': 0.0308, 'eval_runtime': 27.2949, 'eval_samples_per_second': 3.664, 'eval_steps_per_second': 3.664, 'epoch': 0}


Training Loss,Validation Loss,Step
No log,4.273750,0


starting, val loss 4.274


Step,Training Loss,Validation Loss
5,4.003656,4.233534
10,4.083502,4.217039


step 2/10 {'loss': 4.023678779602051, 'grad_norm': 1.48776376247406, 'learning_rate': 0.0001, 'epoch': 0.00017908309455587392}
step 4/10 {'loss': 4.003655910491943, 'grad_norm': 0.9072514176368713, 'learning_rate': 0.0001923879532511287, 'epoch': 0.00035816618911174784}
step 5/10 {'eval_loss': 4.233534336090088, 'eval_model_preparation_time': 0.0308, 'eval_runtime': 17.768, 'eval_samples_per_second': 5.628, 'eval_steps_per_second': 5.628, 'epoch': 0.00044770773638968484}
checkpoint 5: real name 4.172, Unknown 4.188, gap +0.015
step 6/10 {'loss': 4.015936851501465, 'grad_norm': 0.7750663757324219, 'learning_rate': 0.000138268343236509, 'epoch': 0.0005372492836676218}
step 8/10 {'loss': 4.104316711425781, 'grad_norm': 0.8868728280067444, 'learning_rate': 6.173165676349103e-05, 'epoch': 0.0007163323782234957}
step 10/10 {'loss': 4.083502292633057, 'grad_norm': 0.7871803045272827, 'learning_rate': 7.612046748871327e-06, 'epoch': 0.0008954154727793697}
step 10/10 {'eval_loss': 4.21703910827

TrainOutput(global_step=10, training_loss=4.0462181091308596, metrics={'train_runtime': 161.1128, 'train_samples_per_second': 0.497, 'train_steps_per_second': 0.062, 'total_flos': 65672724480000.0, 'train_loss': 4.0462181091308596, 'epoch': 0.0008954154727793697})

## Results

In [8]:
print("val loss:", [(h['step'], round(h['eval_loss'], 3)) for h in trainer.state.log_history if 'eval_loss' in h])
print(f"peak GPU memory: {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")

# typical step time (the middle one, so the eval pauses don't count)
gaps = sorted(b - a for a, b in zip(progress.times, progress.times[1:]))
step_time = gaps[len(gaps) // 2]
print(f"{step_time:.1f} s per step of {STEP} pieces -> the whole pass is ~{len(train) * step_time / STEP / 3600:.1f} h")

val loss: [(5, 4.234), (10, 4.217)]
peak GPU memory: 3.9 GB
5.5 s per step of 8 pieces -> the whole pass is ~17.2 h


In [9]:
# test only: carry on from the last checkpoint for 2 more steps
if TEST_RUN:
    trainer = make_trainer(12)
    trainer.train(resume_from_checkpoint=get_last_checkpoint(RUN_DIR))
    print("now at step", trainer.state.global_step)

Step,Training Loss,Validation Loss
12,3.995279,4.216859


step 12/12 {'loss': 3.9952785968780518, 'grad_norm': 0.8012682795524597, 'learning_rate': 4.8943483704846475e-06, 'epoch': 0.0010744985673352436}
step 12/12 {'eval_loss': 4.216859340667725, 'eval_runtime': 35.8592, 'eval_samples_per_second': 2.789, 'eval_steps_per_second': 2.789, 'epoch': 0.0010744985673352436}
checkpoint 12: real name 4.159, Unknown 4.170, gap +0.011
step 12/12 {'train_runtime': 81.108, 'train_samples_per_second': 0.197, 'train_steps_per_second': 0.025, 'total_flos': 77867581440000.0, 'train_loss': 3.9952785968780518, 'epoch': 0.0010744985673352436}
now at step 12


## Save and try it

In [10]:
model.save_pretrained(FINAL)
del trainer, model
torch.cuda.empty_cache()

base = AutoModelForCausalLM.from_pretrained(MODEL_DIR, dtype=torch.bfloat16).to('cuda')
model = PeftModel.from_pretrained(base, FINAL).eval()

torch.manual_seed(0)
for poet in ['Emily Dickinson', 'John Milton', 'Walt Whitman']:
    inputs = tokenizer(f"Author: {poet}\n\n", return_tensors='pt').to('cuda')
    out = model.generate(**inputs, max_new_tokens=50, do_sample=True, temperature=0.8, top_p=0.95,
                         pad_token_id=tokenizer.eos_token_id)
    print(tokenizer.decode(out[0], skip_special_tokens=True))
    print('-----')

Loading weights: 100%|██████████| 310/310 [00:00<00:00, 331.37it/s]


Author: Emily Dickinson
1.
The house I love is a little house,
Where all my things are safe.
But I can never know
Who in the world I am.
I know that you are in the world,
And that I am in the world,
But I
-----
Author: John Milton
John Milton, born in 1608 and baptized at the age of 11, was a significant figure in the English Renaissance, also known as the Golden Age of England. As a devout and zealous follower of the Church of England
-----
Author: Walt Whitman
AFTERWARD I saw the moon in the clouds. It seemed to me as though the moon had been so long there, since I had first looked at it. I felt a little pity for it, and a little pity for me, for I
-----
